# LeetCode #1478: Allocate Mailboxes

https://leetcode.com/problems/allocate-mailboxes/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^k)$ | $O(n)$ |
| **Optimal: DP + Median Cost Precompute ★** | $O(n^2 \cdot k)$ | $O(n^2)$ |

---

## Understanding the Methods

### Brute Force
Try every subset of positions for $k$ mailboxes and compute total distances — exponential in $k$.

### Optimal: DP + Median Cost Precompute ★
Sort houses. Precompute `cost[i][j]` = minimum total distance to serve houses `i..j` with one mailbox (place it at the median house). Then `dp[i][m]` = minimum cost for the first `i` houses with `m` mailboxes. Transition: try all splits of the last segment.

**Constraints:**
* $1 \leq houses.length \leq 100$
* $1 \leq k \leq houses.length$
* $1 \leq houses[i] \leq 10^4$

## Solutions

### C#

In [ ]:
public class Solution {
    public int MinDistance(int[] houses, int k) {
        int n = houses.Length;
        System.Array.Sort(houses);

        // Precompute cost[i][j]: min total distance to cover houses[i..j] with 1 mailbox
        // Optimal mailbox position for a contiguous segment is the median house
        int[,] cost = new int[n, n];
        for (int i = 0; i < n; i++) {
            for (int j = i; j < n; j++) {
                int mid = (i + j) / 2;
                for (int h = i; h <= j; h++)
                    cost[i, j] += Math.Abs(houses[h] - houses[mid]);
            }
        }

        // dp[i][m] = min cost for first i houses using m mailboxes
        int[,] dp = new int[n + 1, k + 1];
        for (int i = 0; i <= n; i++)
            for (int m = 0; m <= k; m++)
                dp[i, m] = int.MaxValue / 2;
        dp[0, 0] = 0;

        for (int m = 1; m <= k; m++) {
            for (int i = m; i <= n; i++) {
                // Try every possible last segment [j..i-1] covered by the m-th mailbox
                for (int j = m - 1; j < i; j++) {
                    if (dp[j, m - 1] < int.MaxValue / 2)
                        dp[i, m] = Math.Min(dp[i, m], dp[j, m - 1] + cost[j, i - 1]);
                }
            }
        }
        return dp[n, k];
    }
}

### Python

In [ ]:
class Solution:
    def min_distance(self, houses: list[int], k: int) -> int:
        n = len(houses)
        houses.sort()

        # Precompute cost[i][j]: min total distance to cover houses[i..j] with 1 mailbox
        # Optimal single mailbox position is the median
        cost = [[0] * n for _ in range(n)]
        for i in range(n):
            for j in range(i, n):
                mid = (i + j) // 2
                for h in range(i, j + 1):
                    cost[i][j] += abs(houses[h] - houses[mid])

        # dp[i][m] = min cost for first i houses using m mailboxes
        INF = float('inf')
        dp = [[INF] * (k + 1) for _ in range(n + 1)]
        dp[0][0] = 0

        for m in range(1, k + 1):
            for i in range(m, n + 1):
                # Try every last segment [j..i-1] covered by mailbox m
                for j in range(m - 1, i):
                    if dp[j][m - 1] < INF:
                        dp[i][m] = min(dp[i][m], dp[j][m - 1] + cost[j][i - 1])

        return dp[n][k]

### Go

In [ ]:
import "sort"

func minDistance(houses []int, k int) int {
    n := len(houses)
    sort.Ints(houses)

    // Precompute cost[i][j]: optimal single-mailbox total distance for segment i..j
    cost := make([][]int, n)
    for i := range cost { cost[i] = make([]int, n) }
    for i := 0; i < n; i++ {
        for j := i; j < n; j++ {
            mid := (i + j) / 2
            for h := i; h <= j; h++ {
                d := houses[h] - houses[mid]
                if d < 0 { d = -d }
                cost[i][j] += d
            }
        }
    }

    const INF = 1 << 30
    dp := make([][]int, n+1)
    for i := range dp {
        dp[i] = make([]int, k+1)
        for m := range dp[i] { dp[i][m] = INF }
    }
    dp[0][0] = 0

    for m := 1; m <= k; m++ {
        for i := m; i <= n; i++ {
            for j := m - 1; j < i; j++ {
                if dp[j][m-1] < INF {
                    if v := dp[j][m-1] + cost[j][i-1]; v < dp[i][m] { dp[i][m] = v }
                }
            }
        }
    }
    return dp[n][k]
}

### Rust

In [ ]:
impl Solution {
    pub fn min_distance(mut houses: Vec<i32>, k: i32) -> i32 {
        let n = houses.len();
        let k = k as usize;
        houses.sort();

        // Precompute cost[i][j]: optimal single-mailbox total distance for segment i..j
        let mut cost = vec![vec![0i32; n]; n];
        for i in 0..n {
            for j in i..n {
                let mid = (i + j) / 2;
                for h in i..=j {
                    cost[i][j] += (houses[h] - houses[mid]).abs();
                }
            }
        }

        const INF: i32 = i32::MAX / 2;
        let mut dp = vec![vec![INF; k + 1]; n + 1];
        dp[0][0] = 0;

        for m in 1..=k {
            for i in m..=n {
                for j in (m-1)..i {
                    if dp[j][m-1] < INF {
                        let v = dp[j][m-1] + cost[j][i-1];
                        if v < dp[i][m] { dp[i][m] = v; }
                    }
                }
            }
        }
        dp[n][k]
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `houses = [1,4,8,10,20]`, `k = 3`
After sorting: $[1,4,8,10,20]$. Optimal placement: mailboxes at 4 (serves 1,4), 8 (serves 8,10 — distance 2), 20 (serves 20). Total distance: $(3+0) + (0+2) + 0 = 5$.

### 2. Slightly Complex
**Input:** `houses = [2,3,5,12,18]`, `k = 2`
Optimal split: mailbox at 3 serves $[2,3,5]$ (cost $1+0+2=3$), mailbox at 15 serves $[12,18]$ (cost $3+3=6$). Total: $9$.

### 3. Edge Case: Time Factor
**Input:** 100 houses, $k = 100$
Each house gets its own mailbox — distance is always 0. The cost precomputation runs $O(n^2)$ and the DP fills in $O(n^2 \cdot k) = O(10^6)$ cells total.

### 4. Edge Case: Space Factor
**Input:** 100 houses
The cost table is $100 \times 100 = 10{,}000$ integers; the DP table is $101 \times 101 = 10{,}201$ integers. Both fit comfortably in $O(n^2)$ space.

### 5. Almost-Impossible but Plausible
**Input:** `houses = [1,10000]`, `k = 1`
One mailbox must serve both houses. The median position is between them; placed at either house, the total cost is $9999$. The DP correctly returns $9999$.